## FLOOD DETECTION SCENARIO PIPELINE

In [9]:
import digitalhub as dh
PROJECT_NAME = "flood-detection"
proj = dh.get_or_create_project(PROJECT_NAME) # source="git://github.com/scc-digitalhu

### Download Sentinel 2

Register to the open data space copernicus(if not already) and get your credentials.

https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=cdse-public&tab_id=FIiRPJeoiX4

Log the credentials as project secret keys as shown below

In [3]:
# THIS NEED TO BE EXECUTED JUST ONCE
secret0 = proj.new_secret(name="CDSETOOL_ESA_USER", secret_value="esa_username")
secret1 = proj.new_secret(name="CDSETOOL_ESA_PASSWORD", secret_value="esa_password")

In [ ]:
function_s2 = proj.new_function("download_images_s2",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.5",command="python")

### Download Sentinel 1

In [ ]:
function_s1 = proj.new_function("download_images_s1",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.5",command="python")

### Log artifact

The pipeline requires shape files input of river, lakes, and slope.

Log the river shape file. Download the zip file from the [SIAT Portal](https://siat.provincia.tn.it/geonetwork/srv/ita/catalog.search#/metadata/p_TN:df06e63c-d0f3-46c9-8ec2-c25a22c50ef7) and extract the contents inside a folder 'Rivers_TN' and log it as project artifact

In [ ]:
artifact_name='Rivers_TN'
src_path='Rivers_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Log the lakes shape file. Download the zip file from the [SIAT Portal](https://siat.provincia.tn.it/geonetwork/srv/ita/catalog.search#/metadata/p_TN:0f1fdc33-5c71-4c6d-81e7-25eb2ab0e599) and extract the contents inside a folder 'Lakes_TN' and log it as project artifact

In [ ]:
artifact_name='Lakes_TN'
src_path='Lakes_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Log the slope shape file. Download the zip file from the [SIAT Portal](https://webgis.provincia.tn.it/) and extract the contents inside a folder 'Slopes_TN' and log it as project artifact

In [ ]:
artifact_name='Slopes_TN'
src_path='Slopes_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

The resulting datasets will be registered as the project artifact in the datalake under the given names ('Rivers_TN', 'Slopes_TN', 'Lakes_TN').

### Elaboration

In [ ]:
function_rs = proj.new_function("elaborate",kind="container", image="ghcr.io/tn-aixpa/rs-flood-mapping:2.8", code_src="launch.sh")

### Pipeline

In [ ]:
%%writefile "flood_pipeline.py"

from digitalhub_runtime_kfp.dsl import pipeline_context
import datetime

def myhandler(geometry, outputName, floodDate, s1_preFloodDate, s1_postFloodDate, s2_preFloodDate, s2_postFloodDate):
  
    string_dict_data_s1Pre =  """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(s1_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": "sentinel1_GRD_preflood"}"""
    string_dict_data_s1Post = """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s1_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": "sentinel1_GRD_postflood"}"""
    string_dict_data_s2Pre =  """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(s2_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": "sentinel2_pre_flood","preprocess_data_only": "false"}"""
    string_dict_data_s2Post = """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s2_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": "sentinel2_post_flood","preprocess_data_only": "false"}"""
           
    with pipeline_context() as pc:

        s1 = pc.step(name="downloadS1Pre",
                     function="download_images_s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ])

        s2 = pc.step(name="downloadS1Post",
                     function="download_images_s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s1)
        
        s3 = pc.step(name="downloadS2Pre",
                     function="download_images_s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s2)

        s4 = pc.step(name="downloadS2Post",
                     function="download_images_s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s3)

        s5 = pc.step(name="elaborate",
                     function="elaborate",
                     action="job",
                     fs_group='8877',
                     resources={"cpu": {"requests": "3", "limits": "6"},"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/data",
                        "spec": { "size": "200Gi" }
                    }],
                     args=['/shared/launch.sh', 'sentinel1_GRD_preflood', 'sentinel1_GRD_postflood', 'sentinel2_pre_flood', 'sentinel2_post_flood', str(geometry), 'Slopes_TN', 'slope_map25832.tif', 'Lakes_TN', 'idrspacq.shp', 'Rivers_TN', 'cif_pta2022_v.shp', str(outputName), str(floodDate), 'EPSG:25832', "['VV','VH']", '700', '7', '15', '2']
                     ).after(s4)
     


Overwriting flood_pipeline.py


Create workflow using project repo source file

In [ ]:
workflow = proj.new_workflow(
name="pipeline_flood_gitversion",
kind="kfp",
code_src="git+https://<username>:<personal_access_token>@github.com/tn-aixpa/rs-flood-mapping",
handler="src.flood_pipeline:myhandler")

Build workflow

In [ ]:
wfbuild = workflow.run(action="build", wait=True)

Run workflow (Garda)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.644988646837982 45.85539621678084, 10.644988646837982 46.06780100571985, 10.991744628283294 46.06780100571985, 10.991744628283294 45.85539621678084, 10.644988646837982 45.85539621678084))",
    "outputName": "flood_mask_oct_2020",
    "floodDate":"2020-10-02",
    "s1_preFloodDate": "2020-09-25",
    "s1_postFloodDate": "2020-10-09",
    "s2_preFloodDate": "2020-09-12",
    "s2_postFloodDate": "2020-10-22"
    })

Run workflow (Vad di Non)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.640782493560744 46.42790226539165, 11.157037287805009 46.420279834833785, 11.064149949974212 46.03812698432117, 10.558816402631928 46.064237047456984, 10.640782493560744 46.42790226539165))",
    "outputName": "val_di_non_oct_2018",
    "floodDate":"2018-10-27",
    "s1_preFloodDate": "2018-10-20",
    "s1_postFloodDate": "2018-11-04",
    "s2_preFloodDate": "2018-10-07",
    "s2_postFloodDate": "2018-11-17"
    })